In [1]:
!pip install datasets tiktoken tqdm pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 43.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 168.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 343.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 212.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 261.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [datasets]/11 [datasets]ce-hub]


In [2]:
# ============================================================
#  Inner Thinking Transformer (ITT) – GPT-2 scale
#  Architecture : ITT-GPT2 (Corrected Identity Path & Variance)
#  Training     : pure causal LM on BabyLM-2026-Strict-Small
# ============================================================

import os
import math
import time
import json
import random
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
import tiktoken

# ─────────────────────────────────────────────────────────────
# 0.  GLOBAL HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────
VOCAB_SIZE        = 50257
BLOCK_SIZE        = 512
BATCH_SIZE        = 16
# FIX #1: Increased Gradient Accumulation to simulate a ~131k token batch size
GRAD_ACCUM_STEPS  = 16 
EPOCHS            = 10
LEARNING_RATE     = 1e-4
LR_MIN            = LEARNING_RATE * 0.1
WARMUP_STEPS      = 111 #no of steps in 1st epoch
WEIGHT_DECAY      = 0.1
GRAD_CLIP         = 1.0

BLIMP_FAST_PATH   = "fast_eval/blimp_fast"
SUPP_FAST_PATH    = "fast_eval/supplement_fast"


# ─────────────────────────────────────────────────────────────
# 1.  MODEL CONFIG 
# ─────────────────────────────────────────────────────────────

@dataclass
class GPTConfig:
    block_size: int = BLOCK_SIZE
    vocab_size: int = VOCAB_SIZE
    n_layer:    int = 12
    n_head:     int = 12
    n_embd:     int = 768


# ─────────────────────────────────────────────────────────────
# 2.  ARCHITECTURE 
# ─────────────────────────────────────────────────────────────

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        assert config.n_embd % config.n_head == 0

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.RN_FLAGS_SCALE_DOWN = 1

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        head_dim = C // self.n_head
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y


class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu   = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.RN_FLAGS_SCALE_DOWN = 1

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class ITTBlock(nn.Module):
    """
    Inner Thinking Transformer layer (§3 of the ITT paper).
      1. Residual Thinking Connection (RTC) – Eq. 4 / 7
      2. Adaptive Token Routing  (ATR)       – Eq. 5 / 6
      3. Thinking Step Encoding  (φ)
    """

    def __init__(self, config, max_thinking_steps: int = 4, selection_ratio: float = 0.7):
        super().__init__()
        self.config = config
        self.T   = max_thinking_steps
        self.rho = selection_ratio

        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

        self.routers = nn.ModuleList([
            nn.Linear(config.n_embd, 1, bias=False)
            for _ in range(self.T + 1)
        ])

        self.thinking_step_embeddings = nn.Parameter(
            torch.ones(self.T + 1, config.n_embd)
        )
        self.alpha = nn.Parameter(torch.ones(self.T + 1))

        # FIX #2: Zero-init the deeper thinking steps so the model starts stable
        # This prevents variance explosion at initialization.
        with torch.no_grad():
            self.thinking_step_embeddings[1:].zero_()
            self.alpha[1:].zero_()

    def _f(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

    def forward(self, x):
        B, T_seq, C = x.size()

        y_0           = self._f(x)
        phi_0         = self.thinking_step_embeddings[0].view(1, 1, C)
        y_accumulated = y_0 * phi_0
        y_prev        = y_0

        k = max(1, int(self.rho * T_seq))

        for t in range(1, self.T + 1):
            router_logits  = self.routers[t](y_prev).squeeze(-1)
            w_t            = torch.sigmoid(router_logits)

            _, topk_indices = torch.topk(w_t, k, dim=-1, largest=True, sorted=False)
            selection_mask  = torch.zeros(B, T_seq, dtype=torch.bool, device=x.device)
            selection_mask.scatter_(1, topk_indices, True)

            y_full       = self._f(y_prev)
            w_t_expand   = w_t.unsqueeze(-1)
            mask_expand  = selection_mask.unsqueeze(-1)

            # FIX #3: Extract the pure residual (delta) to prevent scaling the identity path
            y_delta = y_full - y_prev

            # Scale ONLY the new information (y_delta), then add it back to y_prev
            y_t_prime = torch.where(
                mask_expand,
                y_prev + (self.alpha[t] * w_t_expand * y_delta),
                y_prev
            )

            step_contribution = torch.where(
                mask_expand,
                y_t_prime,
                torch.zeros_like(y_prev)
            )
            phi_t         = self.thinking_step_embeddings[t].view(1, 1, C)
            y_accumulated = y_accumulated + step_contribution * phi_t
            y_prev        = y_t_prime

        return y_accumulated


class GPT(nn.Module):

    def __init__(self, config: GPTConfig,
                 max_thinking_steps: int = 4,
                 selection_ratio:    float = 0.7):
        super().__init__()
        self.config = config

        blocks = []
        for layer_id in range(config.n_layer):
            if layer_id % 2 == 1:
                blocks.append(ITTBlock(config, max_thinking_steps, selection_ratio))
            else:
                blocks.append(Block(config))

        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            h    = nn.ModuleList(blocks),
            ln_f = nn.LayerNorm(config.n_embd),
        ))

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        total = sum(p.numel() for p in self.parameters())
        print(f"[ITT-GPT2] Total parameters: {total / 1e6:.2f}M")

    def _init_weights(self, module):
        std = 0.02
        if hasattr(module, 'RN_FLAGS_SCALE_DOWN'):
            std *= (2 * self.config.n_layer) ** -0.5
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor,
                targets: torch.Tensor = None,
                bidirectional: bool = False):   # kept for eval-helper compatibility
        B, T = idx.size()
        assert T <= self.config.block_size, \
            f"Sequence length {T} exceeds block_size {self.config.block_size}"

        pos     = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x       = self.transformer.wte(idx) + self.transformer.wpe(pos)

        for block in self.transformer.h:
            x = block(x)

        x      = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1)
            )

        return logits, loss

    def configure_optimizers(self, weight_decay: float, learning_rate: float, device_type: str):
        param_dict     = {n: p for n, p in self.named_parameters() if p.requires_grad}
        decay_params   = [p for p in param_dict.values() if p.dim() >= 2]
        nodecay_params = [p for p in param_dict.values() if p.dim() < 2]
        groups = [
            {'params': decay_params,   'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        num_decay   = sum(p.numel() for p in decay_params)
        num_nodecay = sum(p.numel() for p in nodecay_params)
        print(f"[Optimizer] decayed tensors:    {len(decay_params):4d}  | {num_decay:,} params")
        print(f"[Optimizer] non-decayed tensors:{len(nodecay_params):4d}  | {num_nodecay:,} params")
        fused_ok  = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_ok and ('cuda' in device_type)
        print(f"[Optimizer] fused AdamW: {use_fused}")
        return torch.optim.AdamW(groups, lr=learning_rate,
                                 betas=(0.9, 0.95), eps=1e-8, fused=use_fused)


# ─────────────────────────────────────────────────────────────
# 3.  DATA LOADER
# ─────────────────────────────────────────────────────────────

class DataLoaderLite:

    def __init__(self, B: int, T: int, texts: list, name: str = ""):
        self.B, self.T = B, T
        enc         = tiktoken.get_encoding("gpt2")
        self.eos_id = 50256

        print(f"[DataLoader:{name}] Tokenising {len(texts)} documents with tiktoken …")
        all_ids = []
        for t in texts:
            if t.strip():
                all_ids.extend(enc.encode(t, allowed_special={'<|endoftext|>'}))
                all_ids.append(self.eos_id)

        self.tokens     = torch.tensor(all_ids, dtype=torch.long)
        self.chunk_size = B * T
        self.n_chunks   = (len(self.tokens) - 1) // self.chunk_size
        self.indices    = list(range(self.n_chunks))
        self.pos        = 0
        self._shuffle()

        print(f"[DataLoader:{name}] Total tokens: {len(self.tokens):,} | "
              f"Iterations per epoch: {self.n_chunks:,}")

    def _shuffle(self):
        random.shuffle(self.indices)
        self.pos = 0

    def steps_per_epoch(self) -> int:
        return self.n_chunks

    def next_batch(self):
        if self.pos >= len(self.indices):
            self._shuffle()
        chunk_idx  = self.indices[self.pos]
        self.pos  += 1
        start      = chunk_idx * self.chunk_size
        temp       = self.tokens[start : start + self.chunk_size + 1]
        x = temp[:-1].view(self.B, self.T)
        y = temp[1: ].view(self.B, self.T)
        return x, y


# ─────────────────────────────────────────────────────────────
# 4.  LEARNING RATE SCHEDULER
# ─────────────────────────────────────────────────────────────

def get_lr(it: int, total_steps: int) -> float:
    if it < WARMUP_STEPS:
        return LEARNING_RATE * (it + 1) / WARMUP_STEPS
    if it >= total_steps:
        return LR_MIN
    decay_ratio = (it - WARMUP_STEPS) / (total_steps - WARMUP_STEPS)
    coeff       = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return LR_MIN + coeff * (LEARNING_RATE - LR_MIN)


# ─────────────────────────────────────────────────────────────
# 5.  ITT rho warm-up  (Algorithm 1 getCapacity)
# ─────────────────────────────────────────────────────────────

def get_selection_ratio(global_step: int, target_rho: float = 0.7) -> float:
    if global_step < WARMUP_STEPS:
        return target_rho * (global_step / WARMUP_STEPS)
    return target_rho


# ─────────────────────────────────────────────────────────────
# 6.  EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────

@torch.inference_mode()
def sentence_log_prob(sentence: str, model: GPT, enc,
                      device, autocast_ctx,
                      bidirectional: bool = False) -> float:
    ids = enc.encode(sentence, allowed_special={'<|endoftext|>'})
    if len(ids) <= 1:
        return -1e9
    ids = ids[:model.config.block_size]
    inp = torch.tensor([ids], dtype=torch.long, device=device)
    with autocast_ctx:
        logits, _ = model(inp, targets=None, bidirectional=bidirectional)
    shift_logits = logits[0, :-1, :]
    shift_labels = inp[0, 1:]
    lp = F.log_softmax(shift_logits, dim=-1)
    return lp[torch.arange(len(shift_labels)), shift_labels].sum().item()


def eval_blimp_folder(folder: str, model: GPT, enc,
                      device, autocast_ctx,
                      tag: str = "BLiMP") -> float:
    if not os.path.exists(folder):
        print(f"  [WARN] '{folder}' not found — skipping {tag}.")
        return 0.0
    total, correct = 0, 0
    for fn in sorted(os.listdir(folder)):
        if not fn.endswith('.jsonl'):
            continue
        with open(os.path.join(folder, fn), encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                sg   = item.get('sentence_good')
                sb   = item.get('sentence_bad')
                if sg and sb:
                    lp_g = sentence_log_prob(sg, model, enc, device, autocast_ctx)
                    lp_b = sentence_log_prob(sb, model, enc, device, autocast_ctx)
                    correct += int(lp_g > lp_b)
                    total   += 1
    acc = 100.0 * correct / total if total > 0 else 0.0
    print(f"  [{tag}] {correct}/{total} = {acc:.2f}%")
    return acc


def run_mid_epoch_eval(model: GPT, enc, device, autocast_ctx,
                       epoch: int, half: int):
    model.eval()
    label = f"Epoch {epoch+1}  " + ("— FIRST HALF ↓" if half == 0 else "— SECOND HALF ↓")
    print(f"\n{'='*65}\n  MID-EPOCH ZERO-SHOT EVAL  {label}\n{'='*65}")
    b_acc = 0.00 #skipping this evaln bechmark it takes lot of time 
    s_acc = eval_blimp_folder(SUPP_FAST_PATH,  model, enc, device, autocast_ctx, tag="Supplement-fast")
    print(f"  ➜  BLiMP-fast: {b_acc:.2f}%   Supplement-fast: {s_acc:.2f}%\n{'='*65}\n")
    model.train()
    return b_acc, s_acc


@torch.inference_mode()
def evaluate_val_loss(model: GPT, val_loader: DataLoaderLite,
                      device, autocast_ctx,
                      eval_iters: int = 20) -> float:
    model.eval()
    old_pos     = val_loader.pos
    old_indices = list(val_loader.indices)

    total = 0.0
    for _ in range(eval_iters):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with autocast_ctx:
            _, loss = model(x, y)
        total += loss.item()

    val_loader.indices = old_indices
    val_loader.pos     = old_pos
    model.train()
    return total / eval_iters


# ─────────────────────────────────────────────────────────────
# 7.  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    torch.manual_seed(42)
    random.seed(42)

    device      = 'cuda' if torch.cuda.is_available() else 'cpu'
    device_type = 'cuda' if 'cuda' in device else 'cpu'
    amp_dtype   = torch.bfloat16
    autocast_ctx = torch.autocast(device_type=device_type, dtype=amp_dtype,
                                  enabled=(device_type == 'cuda'))
    if device_type == 'cuda':
        torch.set_float32_matmul_precision('high')
        torch.cuda.manual_seed(42)

    print(f"[Device] Using: {device}")

    enc = tiktoken.get_encoding("gpt2")

    print("\n[Data] Loading BabyLM-2026-Strict-Small …")
    ds       = load_dataset("BabyLM-community/BabyLM-2026-Strict-Small")
    all_text = list(ds['train']['text'])

    split       = int(len(all_text) * 0.95)
    train_texts = all_text[:split]
    val_texts   = all_text[split:]

    train_loader = DataLoaderLite(BATCH_SIZE, BLOCK_SIZE, train_texts, name="train")
    val_loader   = DataLoaderLite(BATCH_SIZE, BLOCK_SIZE, val_texts,   name="val")

    # Due to increased GRAD_ACCUM_STEPS, total optimization steps are reduced
    # So we divide the data loader chunks by GRAD_ACCUM_STEPS to find the true steps per epoch
    chunks_per_epoch = train_loader.steps_per_epoch()
    steps_per_epoch  = chunks_per_epoch // GRAD_ACCUM_STEPS
    total_steps      = steps_per_epoch * EPOCHS
    half_step        = steps_per_epoch // 2

    print(f"[Training] optimizations/epoch={steps_per_epoch:,}  total_optimizations={total_steps:,}")

    cfg   = GPTConfig()
    model = GPT(cfg, max_thinking_steps=4, selection_ratio=0.7).to(device)

    try:
        model = torch.compile(model)
        print("[Model] torch.compile() succeeded")
    except Exception as e:
        print(f"[Model] torch.compile() skipped ({e})")

    optimizer = model.configure_optimizers(WEIGHT_DECAY, LEARNING_RATE, device_type)
    os.makedirs("checkpoints", exist_ok=True)

    model.train()
    global_step = 0
    best_val    = float('inf')

    for epoch in range(EPOCHS):
        train_loader._shuffle()
        first_half_done  = False
        second_half_done = False

        for step in range(steps_per_epoch):
            t0 = time.perf_counter()

            lr = get_lr(global_step, total_steps)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            current_rho = get_selection_ratio(global_step, target_rho=0.7)
            raw_model   = model._orig_mod if hasattr(model, '_orig_mod') else model
            for block in raw_model.transformer.h:
                if isinstance(block, ITTBlock):
                    block.rho = current_rho

            optimizer.zero_grad(set_to_none=True)
            loss_accum = 0.0

            for _ in range(GRAD_ACCUM_STEPS):
                x, y = train_loader.next_batch()
                x, y = x.to(device), y.to(device)
                with autocast_ctx:
                    _, loss = model(x, y)
                scaled      = loss / GRAD_ACCUM_STEPS
                loss_accum += scaled.item()
                scaled.backward()

            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            if device_type == 'cuda':
                torch.cuda.synchronize()

            dt  = (time.perf_counter() - t0) * 1000
            tps = (BATCH_SIZE * BLOCK_SIZE * GRAD_ACCUM_STEPS) / (dt / 1000)

            val_str = ""
            if global_step % 50 == 0:
                vl = evaluate_val_loss(model, val_loader, device, autocast_ctx, eval_iters=20)
                val_str = f"  val={vl:.4f}"
                if vl < best_val:
                    best_val  = vl
                    save_model = model._orig_mod if hasattr(model, '_orig_mod') else model
                    torch.save({
                        'step':                 global_step,
                        'epoch':                epoch,
                        'model_state_dict':     save_model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'config':               cfg,
                        'val_loss':             vl,
                    }, "checkpoints/itt_gpt2_best.pt")

            print(
                f"[E{epoch+1:02d} {step+1:>5d}/{steps_per_epoch} G{global_step:>7d}] "
                f"train={loss_accum:.4f}{val_str}  norm={norm:.3f}  "
                f"lr={lr:.2e}  rho={current_rho:.2f}  dt={dt:6.1f}ms  tok/s={tps:,.0f}"
            )

            if (step + 1) == half_step and not first_half_done:
                first_half_done = True
                run_mid_epoch_eval(model, enc, device, autocast_ctx, epoch, half=0)

            if (step + 1) == steps_per_epoch and not second_half_done:
                second_half_done = True
                run_mid_epoch_eval(model, enc, device, autocast_ctx, epoch, half=1)

            global_step += 1

        save_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        torch.save({
            'epoch':                epoch + 1,
            'step':                 global_step,
            'model_state_dict':     save_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config':               cfg,
        }, f"checkpoints/itt_gpt2_epoch{epoch+1:02d}.pt")


if __name__ == "__main__":
    main()

[Device] Using: cuda



[Data] Loading BabyLM-2026-Strict-Small …


README.md:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

bnc_spoken.train.txt:   0%|          | 0.00/4.06M [00:00<?, ?B/s]

childes.train.txt:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

...

switchboard.train.txt:   0%|          | 0.00/123k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1104106 [00:00<?, ? examples/s]

[DataLoader:train] Tokenising 1048900 documents with tiktoken …
[DataLoader:train] Total tokens: 14,561,988 | Iterations per epoch: 1,777
[DataLoader:val] Tokenising 55206 documents with tiktoken …
[DataLoader:val] Total tokens: 1,941,013 | Iterations per epoch: 236
[Training] optimizations/epoch=111  total_optimizations=1,110
[ITT-GPT2] Total parameters: 124.09M
[Model] torch.compile() succeeded
[Optimizer] decayed tensors:      86  | 123,971,328 params
[Optimizer] non-decayed tensors: 104  | 121,374 params
[Optimizer] fused AdamW: True
[E01     1/111 G      0] train=10.9991  val=10.9090  norm=27.603  lr=9.01e-07  rho=0.00  dt=31715.5ms  tok/s=4,133
[E01     2/111 G      1] train=10.9158  norm=22.645  lr=1.80e-06  rho=0.01  dt=25253.7ms  tok/s=5,190
[E01     3/111 G      2] train=10.7134  norm=26.122  lr=2.70e-06  rho=0.01  dt=19296.2ms  tok/s=6,793
[E01     4/111 G      3] train=10.5064  norm=20.886  lr=3.60e-06  rho=0.02  dt=18629.3ms  tok/s=7,036
[E01     5/111 G      4] train=10.2

W0621 09:20:01.515000 1402 /system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1016] [0/8] torch._dynamo hit config.recompile_limit (8)
W0621 09:20:01.515000 1402 /system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1016] [0/8]    function: 'forward' (/tmp/ipykernel_1402/1669794736.py:237)
W0621 09:20:01.515000 1402 /system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1016] [0/8]    last reason: 0/7: ___as_tensor(self._modules['transformer']._modules['h']._modules['1'].rho).item() == 0.03783783783783784  # (unknown source ___as_tensor(self._modules['transformer']._modules['h']._modules['1'].rho).item(), please file a bug)
W0621 09:20:01.515000 1402 /system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1016] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0621 09:20:01.515000 

[E01     7/111 G      6] train=9.8481  norm=11.088  lr=6.31e-06  rho=0.04  dt=19646.2ms  tok/s=6,672
[E01     8/111 G      7] train=9.6325  norm=10.004  lr=7.21e-06  rho=0.04  dt=1873.6ms  tok/s=69,957
[E01     9/111 G      8] train=9.3919  norm=10.046  lr=8.11e-06  rho=0.05  dt=1812.3ms  tok/s=72,325
[E01    10/111 G      9] train=9.3104  norm=8.106  lr=9.01e-06  rho=0.06  dt=1810.9ms  tok/s=72,381
[E01    11/111 G     10] train=9.3456  norm=6.421  lr=9.91e-06  rho=0.06  dt=1811.0ms  tok/s=72,375
[E01    12/111 G     11] train=9.4497  norm=6.190  lr=1.08e-05  rho=0.07  dt=1812.7ms  tok/s=72,310
[E01    13/111 G     12] train=9.0680  norm=6.395  lr=1.17e-05  rho=0.08  dt=1810.3ms  tok/s=72,402
[E01    14/111 G     13] train=8.3546  norm=10.755  lr=1.26e-05  rho=0.08  dt=1810.7ms  tok/s=72,389
[E01    15/111 G     14] train=8.6448  norm=6.604  lr=1.35e-05  rho=0.09  dt=1810.7ms  tok/s=72,389
[E01    16/111 G     15] train=8.5055  norm=7.078  lr=1.44e-05  rho=0.09  dt=1811.1ms  tok/s=72,

In [3]:
"""
Automated Hyper-Fast Zero-Shot Evaluation Suite for Inner Thinking Transformer (ITT)
Optimizations: torch.inference_mode(), Mixed Precision Autocast, optional torch.compile()
"""

import os
import json
import math
import inspect
from dataclasses import dataclass
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from tqdm.notebook import tqdm

# ==========================================
# 1. CONFIG (Matches pretraining ITT exactly)
# ==========================================
@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 50257
    n_layer:    int = 12
    n_head:     int = 12
    n_embd:     int = 768


# ==========================================
# 2. STANDARD CAUSAL SELF ATTENTION & MLP
# ==========================================
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        assert config.n_embd % config.n_head == 0

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        head_dim = C // self.n_head
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu   = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


# ==========================================
# 3. STANDARD LAYER BLOCK & ITT LAYER BLOCK
# ==========================================
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class ITTBlock(nn.Module):
    def __init__(self, config, max_thinking_steps: int = 4, selection_ratio: float = 0.7):
        super().__init__()
        self.config = config
        self.T   = max_thinking_steps
        self.rho = selection_ratio

        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

        self.routers = nn.ModuleList([
            nn.Linear(config.n_embd, 1, bias=False)
            for _ in range(self.T + 1)
        ])

        self.thinking_step_embeddings = nn.Parameter(torch.ones(self.T + 1, config.n_embd))
        self.alpha = nn.Parameter(torch.ones(self.T + 1))

    def _f(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

    def forward(self, x):
        B, T_seq, C = x.size()

        y_0           = self._f(x)
        phi_0         = self.thinking_step_embeddings[0].view(1, 1, C)
        y_accumulated = y_0 * phi_0
        y_prev        = y_0

        k = max(1, int(self.rho * T_seq))

        for t in range(1, self.T + 1):
            router_logits  = self.routers[t](y_prev).squeeze(-1)
            w_t            = torch.sigmoid(router_logits)

            _, topk_indices = torch.topk(w_t, k, dim=-1, largest=True, sorted=False)
            selection_mask  = torch.zeros(B, T_seq, dtype=torch.bool, device=x.device)
            selection_mask.scatter_(1, topk_indices, True)

            y_full       = self._f(y_prev)
            w_t_expand   = w_t.unsqueeze(-1)
            mask_expand  = selection_mask.unsqueeze(-1)

            y_delta = y_full - y_prev

            y_t_prime = torch.where(
                mask_expand,
                y_prev + (self.alpha[t] * w_t_expand * y_delta),
                y_prev
            )

            step_contribution = torch.where(
                mask_expand,
                y_t_prime,
                torch.zeros_like(y_prev)
            )
            phi_t         = self.thinking_step_embeddings[t].view(1, 1, C)
            y_accumulated = y_accumulated + step_contribution * phi_t
            y_prev        = y_t_prime

        return y_accumulated


# ==========================================
# 4. ITT FULL GPT DEFINITION
# ==========================================
class GPT(nn.Module):
    def __init__(self, config: GPTConfig, max_thinking_steps: int = 4, selection_ratio: float = 0.7):
        super().__init__()
        self.config = config

        blocks = []
        for layer_id in range(config.n_layer):
            if layer_id % 2 == 1:
                blocks.append(ITTBlock(config, max_thinking_steps, selection_ratio))
            else:
                blocks.append(Block(config))

        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            h    = nn.ModuleList(blocks),
            ln_f = nn.LayerNorm(config.n_embd),
        ))

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx: torch.Tensor):
        B, T = idx.size()
        pos  = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x    = self.transformer.wte(idx) + self.transformer.wpe(pos)

        for block in self.transformer.h:
            x = block(x)

        x      = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        return logits, None


# ==========================================
# 5. LOG PROB ENGINE (HYPER-OPTIMIZED)
# ==========================================
def calculate_sentence_log_prob(sentence, model, enc, device, block_size, autocast_type):
    tokens = enc.encode(sentence)
    if len(tokens) <= 1:
        return -99999.0
    if len(tokens) > block_size:
        tokens = tokens[:block_size]
        
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    
    with torch.inference_mode():
        with torch.autocast(device_type=device, dtype=autocast_type):
            logits, _ = model(input_ids)
            shift_logits = logits[0, :-1, :]
            shift_labels = input_ids[0, 1:]
            
            log_softmax_vals = F.log_softmax(shift_logits, dim=-1)
            token_log_probs = log_softmax_vals[torch.arange(len(shift_labels)), shift_labels]
            return token_log_probs.sum().item()


# ==========================================
# 6. GENERIC MINIMAL PAIR HANDLER
# ==========================================
def run_jsonl_minimal_pair(folder_path, key_good, key_bad, model, enc, device, block_size, autocast_type, desc="Evaluating"):
    if not os.path.exists(folder_path):
        print(f"Directory skipped: Path {folder_path} does not exist.")
        return 0.0
        
    total, correct = 0, 0
    files = [f for f in os.listdir(folder_path) if f.endswith(".jsonl")]
    
    for filename in tqdm(files, desc=desc, leave=True):
        with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                s_good = item.get(key_good)
                s_bad = item.get(key_bad)
                
                if s_good and s_bad:
                    if calculate_sentence_log_prob(s_good, model, enc, device, block_size, autocast_type) > \
                       calculate_sentence_log_prob(s_bad, model, enc, device, block_size, autocast_type):
                        correct += 1
                    total += 1
    return (correct / total) * 100 if total > 0 else 0.0


# ==========================================
# 7. EXECUTION ENGINE MAIN
# ==========================================
def main():
    checkpoint_path = "checkpoints/itt_gpt2_best.pt"
    fast_data_root = "fast_eval"
    use_compile = False 
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    autocast_type = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
    
    print(f"🎯 Execution Target Node: {device.upper()} | Mixed Precision Mode: {str(autocast_type).split('.')[-1]}")
    
    enc = tiktoken.get_encoding('gpt2')
    config = GPTConfig()
    model = GPT(config, max_thinking_steps=4, selection_ratio=0.7)
    
    if os.path.exists(checkpoint_path):
        print(f"📦 Loading weights from checkpoint tracker: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
        model.load_state_dict(state_dict, strict=True)
    else:
        print(f"⚠️ Warning: Target weights missing. Continuing tracking with standard init parameters.")
        
    model.to(device)
    model.eval()
    
    if use_compile:
        print("🌀 Compiling graph structures via torch.compile()...")
        model = torch.compile(model)

    if not os.path.exists(fast_data_root):
        print(f"⚠️ Warning: Evaluation folder '{fast_data_root}' can't be resolved.")
        return

    print("⚡ Initiating Automated Hyper-Fast Zero-Shot Evaluation Suite Pipelines...")
    results = {}

    # --- 1. BLiMP Fast ---
    print("\nRunning BLiMP Fast Tasks...")
    blimp_fast_path = os.path.join(fast_data_root, "blimp_fast")
    blimp_acc = run_jsonl_minimal_pair(blimp_fast_path, "sentence_good", "sentence_bad", model, enc, device, config.block_size, autocast_type, desc="BLiMP Fast")
    results["BLiMP Fast"] = blimp_acc
    print(f"👉 BLiMP Fast Grand Accuracy: {blimp_acc:.2f}%\n" + "-" * 50)

    # --- 2. Supplement Fast ---
    print("\nRunning Supplement Fast Tasks...")
    supp_fast_path = os.path.join(fast_data_root, "supplement_fast")
    supp_acc = run_jsonl_minimal_pair(supp_fast_path, "sentence_good", "sentence_bad", model, enc, device, config.block_size, autocast_type, desc="Supplement Fast")
    results["Supplement Fast"] = supp_acc
    print(f"👉 Supplement Fast Grand Accuracy: {supp_acc:.2f}%\n" + "-" * 50)

    # --- 3. EWoK Fast ---
    print("\nInitializing EWoK Fast Evaluation Pipeline...")
    ewok_fast_path = os.path.join(fast_data_root, "ewok_fast", "evaluation_data", "fast_eval", "ewok_fast")
    ewok_total, ewok_correct = 0, 0
    if os.path.exists(ewok_fast_path):
        ewok_files = [f for f in os.listdir(ewok_fast_path) if f.endswith(".jsonl")]
        for fn in tqdm(ewok_files, desc="EWoK Tasks"):
            file_total, file_correct = 0, 0
            with open(os.path.join(ewok_fast_path, fn), 'r', encoding='utf-8') as f:
                for line in f:
                    item = json.loads(line)
                    raw_sentences = item.get('sentences')
                    if raw_sentences and "\t" in raw_sentences:
                        s_good, s_bad = raw_sentences.split("\t", 1)
                    else:
                        s_good = item.get('sentence_good', item.get('sentence_acceptable'))
                        s_bad = item.get('sentence_bad', item.get('sentence_unacceptable'))
                    
                    if s_good and s_bad:
                        if calculate_sentence_log_prob(s_good, model, enc, device, config.block_size, autocast_type) > \
                           calculate_sentence_log_prob(s_bad, model, enc, device, config.block_size, autocast_type):
                            file_correct += 1
                            ewok_correct += 1
                        file_total += 1
                        ewok_total += 1
            file_acc = (file_correct / file_total) * 100 if file_total > 0 else 0.0
            print(f"  ↳ 📊 {fn} Accuracy: {file_acc:.2f}% ({file_correct}/{file_total})")
    ewok_acc = (ewok_correct / ewok_total) * 100 if ewok_total > 0 else 0
    results["EWoK Fast"] = ewok_acc
    print(f"👉 EWoK Fast Grand Accuracy: {ewok_acc:.2f}%\n" + "-" * 50)

    # --- 4. WUGs Fast ---
    print("\nRunning WUGs Fast Tasks...")
    wugs_root = os.path.join(fast_data_root, "wug")
    wugs_total, wugs_correct = 0, 0
    wugs_subfolders = ["wug_adj_nominalization", "wug_past_tense"]
    for sub in wugs_subfolders:
        sub_path = os.path.join(wugs_root, sub)
        if os.path.exists(sub_path):
            sub_total, sub_correct = 0, 0
            for fn in os.listdir(sub_path):
                if fn.endswith(".jsonl"):
                    with open(os.path.join(sub_path, fn), 'r', encoding='utf-8') as f:
                        for line in f:
                            item = json.loads(line)
                            raw_sentences = item.get('sentences')
                            if raw_sentences and "\t" in raw_sentences:
                                s_good, s_bad = raw_sentences.split("\t", 1)
                                if calculate_sentence_log_prob(s_good, model, enc, device, config.block_size, autocast_type) > \
                                   calculate_sentence_log_prob(s_bad, model, enc, device, config.block_size, autocast_type):
                                    sub_correct += 1
                                    wugs_correct += 1
                                sub_total += 1
                                wugs_total += 1
            sub_acc = (sub_correct / sub_total) * 100 if sub_total > 0 else 0.0
            print(f"  ↳ 📊 {sub} Accuracy: {sub_acc:.2f}% ({sub_correct}/{sub_total})")
    wugs_acc = (wugs_correct / wugs_total) * 100 if wugs_total > 0 else 0
    results["WUGs Fast"] = wugs_acc
    print(f"👉 WUGs Fast Grand Accuracy: {wugs_acc:.2f}%\n" + "-" * 50)

    # --- 5. Self-Paced Reading Fast ---
    print("\nRunning Self-Paced Reading Fast Surprisals...")
    reading_file = os.path.join(fast_data_root, "reading", "reading_data.csv")
    if os.path.exists(reading_file):
        df = pd.read_csv(reading_file)
        surprisals = []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Reading Tasks"):
            sent = str(row['sentence'])
            word = str(row['word'])
            full_prob = calculate_sentence_log_prob(sent, model, enc, device, config.block_size, autocast_type)
            context_sent = sent.split(word)[0].strip()
            context_prob = calculate_sentence_log_prob(context_sent, model, enc, device, config.block_size, autocast_type) if context_sent else 0.0
            surprisal = -(full_prob - context_prob) / math.log(2)
            surprisals.append(surprisal)
        df['surprisal'] = surprisals
        correlation = df['surprisal'].corr(df['self_paced_reading_time'])
        reading_delta = abs(correlation * 100)
        results["Reading Fast Delta"] = reading_delta
        print(f"👉 Self-Paced Reading Fast R^2 Delta: {reading_delta:.2f}")
    print("-" * 50)

    # --- 6. Entity Tracking Fast ---
    print("\nRunning Entity Tracking Fast Tasks...")
    entity_path = os.path.join(fast_data_root, "entity_tracking_fast")
    ent_total, ent_correct = 0, 0
    if os.path.exists(entity_path):
        ent_files = [f for f in os.listdir(entity_path) if f.endswith(".jsonl")]
        for fn in tqdm(ent_files, desc="Entity Fast"):
            file_total, file_correct = 0, 0
            with open(os.path.join(entity_path, fn), 'r', encoding='utf-8') as f:
                for line in f:
                    item = json.loads(line)
                    prefix = item.get('input_prefix', '')
                    options = item.get('options', [])
                    if prefix and len(options) == 5:
                        log_probs = [calculate_sentence_log_prob(f"{prefix}{opt}", model, enc, device, config.block_size, autocast_type) for opt in options]
                        if log_probs[0] == max(log_probs):
                            file_correct += 1
                            ent_correct += 1
                        file_total += 1
                        ent_total += 1
            file_acc = (file_correct / file_total) * 100 if file_total > 0 else 0.0
            print(f"  ↳ 📊 {fn} Accuracy: {file_acc:.2f}% ({file_correct}/{file_total})")
    entity_acc = (ent_correct / ent_total) * 100 if ent_total > 0 else 0
    results["Entity Tracking Fast"] = entity_acc
    print(f"👉 Entity Tracking Fast Grand Accuracy: {entity_acc:.2f}%\n" + "-" * 50)

    # --- Summary Block ---
    print("\n" + "="*60 + "\n⚡ FAST EVALUATION RESULTS SUMMARY\n" + "="*60)
    for task, score in results.items():
        print(f" 🔹 {task:<30s}: {score:.2f}%" if "Delta" not in task else f" 🔹 {task:<30s}: {score:.2f}")
    print("="*60 + "\n")

if __name__ == "__main__":
    main()

🎯 Execution Target Node: CUDA | Mixed Precision Mode: bfloat16
📦 Loading weights from checkpoint tracker: checkpoints/itt_gpt2_best.pt
⚡ Initiating Automated Hyper-Fast Zero-Shot Evaluation Suite Pipelines...

Running BLiMP Fast Tasks...


BLiMP Fast:   0%|          | 0/67 [00:00<?, ?it/s]

👉 BLiMP Fast Grand Accuracy: 58.28%
--------------------------------------------------

Running Supplement Fast Tasks...


Supplement Fast:   0%|          | 0/5 [00:00<?, ?it/s]

👉 Supplement Fast Grand Accuracy: 56.40%
--------------------------------------------------

Initializing EWoK Fast Evaluation Pipeline...


...

  ↳ 📊 social-interactions.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 physical-relations.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 quantitative-properties.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 spatial-relations.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 agent-properties.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 material-properties.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 physical-interactions.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 material-dynamics.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 social-relations.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 physical-dynamics.jsonl Accuracy: 0.00% (0/0)
  ↳ 📊 social-properties.jsonl Accuracy: 0.00% (0/0)
👉 EWoK Fast Grand Accuracy: 0.00%
--------------------------------------------------

Running WUGs Fast Tasks...
  ↳ 📊 wug_adj_nominalization Accuracy: 36.00% (72/200)
  ↳ 📊 wug_past_tense Accuracy: 100.00% (50/50)
👉 WUGs Fast Grand Accuracy: 48.80%
--------------------------------------------------

Running Self-Paced Reading Fast Surprisals...


Reading Tasks:   0%|          | 0/1726 [00:00<?, ?it/s]

👉 Self-Paced Reading Fast R^2 Delta: 6.07
--------------------------------------------------

Running Entity Tracking Fast Tasks...


Entity Fast:   0%|          | 0/1 [00:00<?, ?it/s]

  ↳ 📊 regular.jsonl Accuracy: 19.19% (605/3152)
👉 Entity Tracking Fast Grand Accuracy: 19.19%
--------------------------------------------------

⚡ FAST EVALUATION RESULTS SUMMARY
 🔹 BLiMP Fast                    : 58.28%
 🔹 Supplement Fast               : 56.40%
 🔹 EWoK Fast                     : 0.00%
 🔹 WUGs Fast                     : 48.80%
 🔹 Reading Fast Delta            : 6.07
 🔹 Entity Tracking Fast          : 19.19%



In [5]:
import os
import json
import math
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

# ==========================================
# 1. EVALUATION WORKING HANDLERS (HYPER-OPTIMIZED)
# ==========================================
def calculate_sentence_log_prob(sentence, model, enc, device, block_size, autocast_type):
    tokens = enc.encode(sentence)
    if len(tokens) <= 1:
        return -99999.0
    if len(tokens) > block_size:
        tokens = tokens[:block_size]
        
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    
    with torch.inference_mode():
        with torch.autocast(device_type=device if "cuda" in str(device) else "cpu", dtype=autocast_type):
            logits, _ = model(input_ids)
            shift_logits = logits[0, :-1, :]
            shift_labels = input_ids[0, 1:]
            
            log_softmax_vals = F.log_softmax(shift_logits, dim=-1)
            token_log_probs = log_softmax_vals[torch.arange(len(shift_labels)), shift_labels]
            
            return token_log_probs.sum().item()

def run_jsonl_minimal_pair(folder_path, key_good, key_bad, model, enc, device, block_size, autocast_type, desc="Evaluating"):
    if not os.path.exists(folder_path):
        print(f"Directory skipped: Path {folder_path} does not exist.")
        return 0.0
        
    total, correct = 0, 0
    files = [f for f in os.listdir(folder_path) if f.endswith(".jsonl")]
    
    for filename in tqdm(files, desc=desc, leave=True):
        with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                s_good = item.get(key_good)
                s_bad = item.get(key_bad)
                
                if s_good and s_bad:
                    prob_good = calculate_sentence_log_prob(s_good, model, enc, device, block_size, autocast_type)
                    prob_bad = calculate_sentence_log_prob(s_bad, model, enc, device, block_size, autocast_type)
                    if prob_good > prob_bad:
                        correct += 1
                    total += 1
    return (correct / total) * 100 if total > 0 else 0.0

# ==========================================
# 2. FULL ZERO-SHOT EVALUATION RUNNER
# ==========================================
def run_full_evaluation_suite(model, enc, device, block_size, autocast_type, data_root="zero shot"):
    """
    Executes the comprehensive, un-filtered evaluation tracks on the ITT model architecture.
    """
    if not os.path.exists(data_root):
        print(f"⚠️ Warning: Evaluation directory path context cannot be found at '{data_root}'. "
              f"Update this path variable to match your uploaded workspace dataset cluster.")
        return

    # Set model to evaluation state to secure dropout/layer normalization tracks
    model.eval()
    print(f"🚀 Initiating Zero-Shot Full Evaluation Suite on Device: {str(device).upper()}")
    print("-" * 60)
    
    results = {}

    # --- 1. BLiMP Full ---
    blimp_path = os.path.join(data_root, "blimp_filtered")
    print("\nRunning Full BLiMP Tracks...")
    blimp_acc = run_jsonl_minimal_pair(blimp_path, "sentence_good", "sentence_bad", model, enc, device, block_size, autocast_type, desc="BLiMP Tasks")
    results["BLiMP Full"] = blimp_acc
    print(f"👉 BLiMP Accuracy: {blimp_acc:.2f}%\n" + "-"*50)

    # --- 2. BLiMP Supplement Full ---
    supp_path = os.path.join(data_root, "supplement_filtered")
    print("\nRunning Full Supplement Tracks...")
    supp_acc = run_jsonl_minimal_pair(supp_path, "sentence_good", "sentence_bad", model, enc, device, block_size, autocast_type, desc="Supplement Tasks")
    results["BLiMP Supplement Full"] = supp_acc
    print(f"👉 BLiMP Supplement Accuracy: {supp_acc:.2f}%\n" + "-"*50)

    # --- 3. COMPS Full ---
    comps_path = os.path.join(data_root, "comps")
    print("\nRunning Full COMPS Tracks...")
    comps_total, comps_correct = 0, 0
    if os.path.exists(comps_path):
        comps_files = [f for f in os.listdir(comps_path) if f.endswith(".jsonl")]
        for fn in tqdm(comps_files, desc="COMPS Tasks"):
            with open(os.path.join(comps_path, fn), 'r', encoding='utf-8') as f:
                for line in f:
                    item = json.loads(line)
                    s_good = f"{item['prefix_acceptable']} {item['property_phrase']}"
                    s_bad = f"{item['prefix_unacceptable']} {item['property_phrase']}"
                    
                    prob_good = calculate_sentence_log_prob(s_good, model, enc, device, block_size, autocast_type)
                    prob_bad = calculate_sentence_log_prob(s_bad, model, enc, device, block_size, autocast_type)
                    if prob_good > prob_bad:
                        comps_correct += 1
                    comps_total += 1
        comps_acc = (comps_correct / comps_total) * 100 if comps_total > 0 else 0
        results["COMPS Full"] = comps_acc
        print(f"👉 COMPS Accuracy: {comps_acc:.2f}%\n" + "-"*50)

    # --- 4. WUGs Full (Updated with tab-separated splitting logic) ---
    print("\nInitializing WUGs Evaluation Pipeline...")
    wugs_root = os.path.join(data_root, "wugs")
    wugs_total, wugs_correct = 0, 0
    wugs_subfolders = ["wug_adj_nominalization", "wug_past_tense"]
    wugs_sub_acc = {}

    for sub in tqdm(wugs_subfolders, desc="WUGs Categories"):
        sub_path = os.path.join(wugs_root, sub)
        if os.path.exists(sub_path):
            sub_total, sub_correct = 0, 0
            for fn in os.listdir(sub_path):
                if fn.endswith(".jsonl"):
                    with open(os.path.join(sub_path, fn), 'r', encoding='utf-8') as f:
                        for line in f:
                            item = json.loads(line)
                            raw_sentences = item.get('sentences')
                            
                            if raw_sentences and "\t" in raw_sentences:
                                s_good, s_bad = raw_sentences.split("\t", 1)
                                
                                prob_good = calculate_sentence_log_prob(s_good, model, enc, device, block_size, autocast_type)
                                prob_bad = calculate_sentence_log_prob(s_bad, model, enc, device, block_size, autocast_type)
                                if prob_good > prob_bad:
                                    sub_correct += 1
                                    wugs_correct += 1
                                sub_total += 1
                                wugs_total += 1
            
            sub_acc = (sub_correct / sub_total) * 100 if sub_total > 0 else 0.0
            wugs_sub_acc[sub] = sub_acc
            print(f"  ↳ 📊 {sub} Accuracy: {sub_acc:.2f}% ({sub_correct}/{sub_total})")

    wugs_acc = (wugs_correct / wugs_total) * 100 if wugs_total > 0 else 0
    results["WUGs Full"] = wugs_acc
    print(f"\n👉 Final Macro WUGs Average Accuracy: {wugs_acc:.2f}%\n" + "-"*50)

    # --- 5. Self-Paced Reading Full (Updated) ---
    print("\nInitializing Self-Paced Reading Surprisal Pipeline...")
    reading_file = os.path.join(data_root, "reading", "reading_data.csv")
    if os.path.exists(reading_file):
        df = pd.read_csv(reading_file)
        surprisals = []
        
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Reading Surprisals"):
            sent = str(row['sentence'])
            word = str(row['word'])
            
            full_prob = calculate_sentence_log_prob(sent, model, enc, device, block_size, autocast_type)
            context_sent = sent.split(word)[0].strip()
            context_prob = calculate_sentence_log_prob(context_sent, model, enc, device, block_size, autocast_type) if context_sent else 0.0
            
            surprisal = -(full_prob - context_prob) / math.log(2)
            surprisals.append(surprisal)
            
        df['surprisal'] = surprisals
        correlation = df['surprisal'].corr(df['self_paced_reading_time'])
        reading_delta = abs(correlation * 100) if not math.isnan(correlation) else 0.0
        results["Self-Paced Reading Delta"] = reading_delta
        print(f"👉 Self-Paced Reading R^2 Delta Value: {reading_delta:.2f}\n" + "-"*50)

    # --- 6. Entity Tracking Full (Updated with 5-choice evaluation tracker) ---
    print("\nInitializing Entity Tracking Evaluation Pipeline...")
    entity_path = os.path.join(data_root, "entity_tracking")
    ent_total, ent_correct = 0, 0
    entity_sub_acc = {}

    if os.path.exists(entity_path):
        ent_files = [f for f in os.listdir(entity_path) if f.endswith(".jsonl")]
        for fn in tqdm(ent_files, desc="Entity Tasks"):
            file_total, file_correct = 0, 0
            with open(os.path.join(entity_path, fn), 'r', encoding='utf-8') as f:
                for line in f:
                    item = json.loads(line)
                    prefix = item.get('input_prefix', '')
                    options = item.get('options', [])
                    
                    if prefix and len(options) == 5:
                        log_probs = []
                        for opt in options:
                            full_sentence = f"{prefix}{opt}"
                            log_probs.append(calculate_sentence_log_prob(full_sentence, model, enc, device, block_size, autocast_type))
                        
                        if log_probs[0] == max(log_probs):
                            file_correct += 1
                            ent_correct += 1
                        file_total += 1
                        ent_total += 1
            
            file_acc = (file_correct / file_total) * 100 if file_total > 0 else 0.0
            entity_sub_acc[fn] = file_acc
            print(f"  ↳ 📊 {fn} Accuracy: {file_acc:.2f}% ({file_correct}/{file_total})")

    entity_acc = (ent_correct / ent_total) * 100 if ent_total > 0 else 0
    results["Entity Tracking Full"] = entity_acc
    print(f"\n👉 Final Macro Entity Tracking Average Accuracy: {entity_acc:.2f}%\n" + "-"*50)

    # --- Summary Block ---
    print("\n" + "="*60 + "\n📊 FULL EVALUATION RESULTS SUMMARY\n" + "="*60)
    for task, score in results.items():
        print(f" 🔹 {task:<30s}: {score:.2f}%" if "Delta" not in task else f" 🔹 {task:<30s}: {score:.2f}")
    print("="*60 + "\n")
# ==========================================
# 3. RUNTIME EXECUTION ENVIRONMENT DEFINITION
# ==========================================

# 1. System Configs
CHECKPOINT_PATH = "checkpoints/itt_gpt2_best.pt"  # Update with your target checkpoint file
DATA_ROOT = "zero shot"                           # Cloud workspace directory
BLOCK_SIZE = 512

# 2. Hardware Engine Strategy
device = "cuda" if torch.cuda.is_available() else "cpu"
autocast_type = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

print(f"🎯 Execution Target Node: {device.upper()}")
print(f"⚡ Mixed Precision Mode:  {str(autocast_type).split('.')[-1]}")
print("-" * 60)

# 3. Initialize Tokenizer Tracker
enc = tiktoken.get_encoding('gpt2')

# 4. Instantiate Newer ITT Model Configuration Architecture
config = GPTConfig(block_size=BLOCK_SIZE) # Matches vocabulary and core pretrain hyper-parameters
model = GPT(config, max_thinking_steps=4, selection_ratio=0.7)

# 5. Load Weights Safe Track State Dictionary 
if os.path.exists(CHECKPOINT_PATH):
    print(f"📦 Loading weights from checkpoint tracker: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
else:
    print(f"⚠️ Warning: Target weights missing at '{CHECKPOINT_PATH}'. Continuing tracking with standard init parameters.")

# Move model parameter weights to targeted device architecture
model.to(device)

# 6. Execute Full Zero-Shot Suite Pipelines
run_full_evaluation_suite(
    model=model,
    enc=enc,
    device=device,
    block_size=BLOCK_SIZE,
    autocast_type=autocast_type,
    data_root=DATA_ROOT
)

🎯 Execution Target Node: CUDA
⚡ Mixed Precision Mode:  bfloat16
------------------------------------------------------------
📦 Loading weights from checkpoint tracker: checkpoints/itt_gpt2_best.pt
🚀 Initiating Zero-Shot Full Evaluation Suite on Device: CUDA
------------------------------------------------------------

Running Full BLiMP Tracks...


BLiMP Tasks:   0%|          | 0/67 [00:00<?, ?it/s]

👉 BLiMP Accuracy: 58.34%
--------------------------------------------------

Running Full Supplement Tracks...


Supplement Tasks:   0%|          | 0/5 [00:00<?, ?it/s]

👉 BLiMP Supplement Accuracy: 68.38%
--------------------------------------------------

Running Full COMPS Tracks...


...

  ↳ 📊 wug_adj_nominalization Accuracy: 36.00% (72/200)
  ↳ 📊 wug_past_tense Accuracy: 100.00% (50/50)

👉 Final Macro WUGs Average Accuracy: 48.80%
--------------------------------------------------

Initializing Self-Paced Reading Surprisal Pipeline...


Reading Surprisals:   0%|          | 0/1726 [00:00<?, ?it/s]

👉 Self-Paced Reading R^2 Delta Value: 6.07
--------------------------------------------------

Initializing Entity Tracking Evaluation Pipeline...


Entity Tasks:   0%|          | 0/3 [00:00<?, ?it/s]

  ↳ 📊 ambiref.jsonl Accuracy: 15.57% (502/3224)
  ↳ 📊 move_contents.jsonl Accuracy: 19.09% (593/3107)
  ↳ 📊 regular.jsonl Accuracy: 19.19% (605/3152)

👉 Final Macro Entity Tracking Average Accuracy: 17.93%
--------------------------------------------------

📊 FULL EVALUATION RESULTS SUMMARY
 🔹 BLiMP Full                    : 58.34%
 🔹 BLiMP Supplement Full         : 68.38%
 🔹 COMPS Full                    : 49.54%
 🔹 WUGs Full                     : 48.80%
 🔹 Self-Paced Reading Delta      : 6.07
 🔹 Entity Tracking Full          : 17.93%



In [6]:
print("ad")

ad
